# Notebook 4: Principal Component Analysis (PCA)
## Federal Reserve Interest Rate Prediction

**Objectives:**
- Reduce 8 raw features to principal components
- Identify the most informative linear combinations
- Visualize economic regime structure in lower dimensions
- Determine minimum components needed to retain 95% variance


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#E91E63','#795548','#607D8B','#FF5722']
sns.set_palette(PALETTE)

DATA_PATH = r"d:/Projects/ML website/ML-Project/App/Tabs/Datasets/finaldataset.csv"
OUT_PATH  = r"d:/Projects/ML website/ML-Project/ml_analysis/outputs"

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


In [ ]:
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').ffill().bfill().reset_index(drop=True)
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].clip(df[col].quantile(0.01), df[col].quantile(0.99))

FEATURES = ['ConsumerPriceIndexAllItems','GDP','InflationConsumerPrice',
            'MedianConsumerPriceIndex','RealGDP','RealGDPPerCapita',
            'RealPotentialGDP','UnemployemenrRate']
X = df[FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Data shape: {X_scaled.shape}")


## 1. Full PCA — Explained Variance

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)
explained   = pca_full.explained_variance_ratio_
cumulative  = np.cumsum(explained)

print("Explained variance per component:")
for i, (ev, cv) in enumerate(zip(explained, cumulative)):
    print(f"  PC{i+1}: {ev*100:.2f}% | Cumulative: {cv*100:.2f}%")

n_95 = np.argmax(cumulative >= 0.95) + 1
print(f"\nComponents for 95% variance: {n_95}")
print(f"Components for 99% variance: {np.argmax(cumulative >= 0.99)+1}")


In [ ]:
# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
comps = range(1, len(explained)+1)
axes[0].bar(comps, explained*100, color='#2196F3', alpha=0.8, label='Individual')
axes[0].plot(comps, cumulative*100, 'ro-', linewidth=2, markersize=7, label='Cumulative')
axes[0].axhline(95, color='green', linestyle='--', linewidth=1.5, label='95% threshold')
axes[0].set_xlabel('Component'); axes[0].set_ylabel('Variance (%)')
axes[0].set_title('PCA Scree Plot', fontweight='bold')
axes[0].legend(); axes[0].set_xticks(comps)

# Cumulative
axes[1].fill_between(comps, cumulative*100, alpha=0.2, color='#2196F3')
axes[1].plot(comps, cumulative*100, 'b-o', linewidth=2, markersize=8)
for i, cv in enumerate(cumulative):
    axes[1].annotate(f'{cv*100:.1f}%', (i+1, cv*100), xytext=(0,8),
                     textcoords='offset points', ha='center', fontsize=8)
axes[1].axhline(95, color='red', linestyle='--', label='95%')
axes[1].axhline(99, color='green', linestyle='--', label='99%')
axes[1].set_xlabel('Components'); axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].legend(); axes[1].set_xticks(comps)
plt.tight_layout(); plt.show()


## 2. PCA Loadings Analysis

In [ ]:
pca_3 = PCA(n_components=3)
X_3d  = pca_3.fit_transform(X_scaled)

loadings = pd.DataFrame(
    pca_3.components_.T,
    index=FEATURES,
    columns=['PC1','PC2','PC3']
)
print("PCA Loadings (contribution of each feature to each PC):")
display(loadings.round(4))

# Loadings heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(loadings, annot=True, fmt='.3f', cmap='RdBu_r', ax=ax,
            linewidths=0.5, center=0)
ax.set_title('PCA Loadings Heatmap (Top 3 Components)', fontweight='bold')
plt.tight_layout(); plt.show()


## 3. PCA Biplot — Visualizing Data in PC Space

In [ ]:
# Create rate direction for coloring
df['RateDirection'] = 'No_Change'
df['RateChange'] = df['FEDRates'].diff()
df.loc[df['RateChange'] > 0.05,  'RateDirection'] = 'Increase'
df.loc[df['RateChange'] < -0.05, 'RateDirection'] = 'Decrease'
color_map = {'Increase':0, 'No_Change':1, 'Decrease':2}
color_vals = df['RateDirection'].map(color_map).values

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
# PC1 vs PC2
sc = axes[0].scatter(X_3d[:,0], X_3d[:,1], c=color_vals[:len(X_3d)],
                     cmap='RdYlGn', alpha=0.5, s=20)
for i, feat in enumerate(FEATURES):
    scale = 3
    axes[0].arrow(0, 0, pca_3.components_[0,i]*scale, pca_3.components_[1,i]*scale,
                  head_width=0.08, fc='#333', ec='#333', alpha=0.8)
    axes[0].text(pca_3.components_[0,i]*scale*1.2, pca_3.components_[1,i]*scale*1.2,
                 feat[:10], fontsize=7)
axes[0].set_xlabel(f"PC1 ({explained[0]*100:.1f}% var)")
axes[0].set_ylabel(f"PC2 ({explained[1]*100:.1f}% var)")
axes[0].set_title('PCA Biplot (PC1 vs PC2)', fontweight='bold')
plt.colorbar(sc, ax=axes[0], label='0=Inc, 1=No, 2=Dec')

# PC1 vs PC3
axes[1].scatter(X_3d[:,0], X_3d[:,2], c=color_vals[:len(X_3d)], cmap='RdYlGn', alpha=0.5, s=20)
axes[1].set_xlabel(f"PC1 ({explained[0]*100:.1f}% var)")
axes[1].set_ylabel(f"PC3 ({explained[2]*100:.1f}% var)")
axes[1].set_title('PCA Biplot (PC1 vs PC3)', fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# PC scores over time
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
dates = df['date'][:len(X_3d)]
for i, (ax, title) in enumerate(zip(axes, ['PC1 — Economic Scale',
                                            'PC2 — Inflation Dynamics',
                                            'PC3 — Labor Market'])):
    ax.plot(dates, X_3d[:,i], color=PALETTE[i], linewidth=1)
    ax.fill_between(dates, X_3d[:,i], alpha=0.1, color=PALETTE[i])
    ax.set_title(title, fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.5)
fig.suptitle('Principal Component Scores Over Time', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## Summary
- PC1 (57.3%) captures overall economic scale (GDP dominates)
- PC2 (18.7%) captures inflation dynamics
- PC3 (13.7%) captures labor market
- 4 components needed for 95% variance — strong multicollinearity among GDP features